# CC1e0pi Selection Analysis
**Pandora (legacy) | BNB MC Overlays | No trigger**

1. Setup and style
2. Load data and define cuts
3. Sanity check: reproduce efficiency curve
4. Cut flow table
5. Electron KE threshold sweep (Pareto map)
6. Individual cut sweeps with Punzi FOM
7. Cut ordering study
8. 2D efficiency/purity maps
9. Neutrino energy resolution and regression

In [ ]:
import uproot
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.colors import Normalize
from scipy.ndimage import gaussian_filter
from scipy.stats import linregress
from scipy.optimize import curve_fit
import warnings
from cmcrameri import cm

warnings.filterwarnings('ignore')

# ── Style ─────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family':         'serif',
    'font.serif':          ['Times New Roman','DejaVu Serif'],
    'mathtext.fontset':    'cm',
    'axes.labelsize':      14,
    'axes.titlesize':      13,
    'axes.titlepad':       10,
    'xtick.labelsize':     11,
    'ytick.labelsize':     11,
    'legend.fontsize':     10,
    'legend.framealpha':   0.88,
    'legend.edgecolor':    '#cccccc',
    'legend.fancybox':     True,
    'figure.dpi':          140,
    'figure.facecolor':    'white',
    'axes.facecolor':      '#f9f9f9',
    'axes.edgecolor':      '#444444',
    'axes.linewidth':      1.3,
    'axes.grid':           True,
    'grid.color':          '#dddddd',
    'grid.linewidth':      0.7,
    'xtick.direction':     'in',
    'ytick.direction':     'in',
    'xtick.top':           True,
    'ytick.right':         True,
    'xtick.minor.visible': True,
    'ytick.minor.visible': True,
    'xtick.major.size':    5,
    'ytick.major.size':    5,
    'xtick.minor.size':    2.5,
    'ytick.minor.size':    2.5,
    'lines.linewidth':     2,
})

CRIMSON  = '#DC143C'
GOLD     = '#DAA520'
BLUE     = '#3a7ebf'
MANAGUA  = cm.managua
SEL_COLS = ['#111111','#8c1d18','#f26b6c','#f39c12',
            '#3f8f3f','#61d6d6','#4257f5','#c13fd4']

def band_plot(ax, x, y, yerr, color, label='', lw=2, ms=6,
              alpha_band=0.18, fmt='o', step=False):
    """Line + marker + shaded error band. Cleaner than individual caps."""
    if np.ndim(yerr)==1:
        ylo, yhi = y-yerr, y+yerr
    else:
        ylo, yhi = y-yerr[0], y+yerr[1]
    if step:
        ax.fill_between(x, ylo, yhi, color=color, alpha=alpha_band,
                        step='mid', zorder=2)
    else:
        ax.fill_between(x, ylo, yhi, color=color, alpha=alpha_band, zorder=2)
    ax.plot(x, y, fmt+'-' if fmt else '-', color=color, lw=lw,
            ms=ms, label=label, zorder=4,
            markeredgecolor='white', markeredgewidth=0.5)

def vline_base(ax, val, color=CRIMSON, label=None):
    ax.axvline(val, ls='--', lw=1.8, color=color, alpha=0.85,
               label=label or f'Baseline ({val})', zorder=6)

def vline_opt(ax, val, color=GOLD, label=None):
    ax.axvline(val, ls=':', lw=1.8, color=color, alpha=0.9,
               label=label or f'FOM opt. ({val:.2f})', zorder=6)

def savefig(fig, fname):
    fig.savefig(fname, dpi=150, bbox_inches='tight')
    print(f'Saved {fname}')

print('Setup complete.')

## 1. Load data
---
Data first run processe through macro: `/exp/icarus/app/users/sdey2/NuGraphReco/cc1e0pi_v3/src/CC1e0piSelection_MakeAll.C` 

This takes ~ 5 hours on gpvms.

In [ ]:
ROOT_FILE = '/exp/icarus/app/users/sdey2/NuGraphReco/cc1e0pi_v3/output/root/CC1e0piSelection_All.root'   

df = uproot.open(ROOT_FILE)['ntuple/ntuple'].arrays(library='pd')

# replace sentinels with NaN
for col in df.columns:
    df[col] = df[col].replace([-5., -9999.], np.nan)

print(f'Slices: {len(df):,}  |  Branches: {len(df.columns)}')

## 2. Baseline cuts
---
The nominal cuts used for NuMI analysis to provide a reference point when comparing BNB.

In [ ]:
THRESHOLD_E        = 0.200
SHOWER_ENERGY_MIN  = 0.200
SHOWER_DEDX_MAX    = 3.500
SHOWER_ANGLE_MAX   = 15.0
SHOWER_CONVGAP_MAX = 4.0
MUON_CHI2_MU_MAX   = 30.0
MUON_CHI2_P_MIN    = 80.0

def signal_mask(df, ke=THRESHOLD_E):
    return (df['is_signal']==1) & (df['true_electron_ke'].fillna(0) > ke)

def c_presel(df):             return (df['pass_not_clear_cosmic']==1) & (df['pass_vertex_fv']==1)
def c_flash(df):              return df['pass_flash_match']==1
def c_shower(df, v=SHOWER_ENERGY_MIN):   return df['shower_energy'].fillna(0) > v
def c_dedx(df, v=SHOWER_DEDX_MAX):      return (df['shower_avail_dedx'].fillna(99)<v) & (df['shower_avail_dedx'].fillna(0)>0)
def c_angle(df, v=SHOWER_ANGLE_MAX):    return df['shower_open_angle'].fillna(99) < v
def c_gap(df, v=SHOWER_CONVGAP_MAX):    return df['shower_conv_gap'].fillna(99) < v
def c_proton(df):             return df['n_protons'].fillna(0) >= 1
def c_0pi(df):                return df['n_other_particles'].fillna(99) == 0
def c_muon(df, v=MUON_CHI2_MU_MAX):
    return ~((df['longest_trk_chi2_muon'].fillna(99)<v) & (df['longest_trk_chi2_proton'].fillna(0)>MUON_CHI2_P_MIN))

def sel_presel(df):   return c_presel(df)
def sel_flash(df):    return sel_presel(df)  & c_flash(df)
def sel_shower(df):   return sel_flash(df)   & c_shower(df)
def sel_dedx(df):     return sel_shower(df)  & c_dedx(df)
def sel_anglegap(df): return sel_dedx(df)    & c_angle(df) & c_gap(df)
def sel_proton(df):   return sel_anglegap(df) & c_proton(df)
def sel_0pi(df):      return sel_proton(df)  & c_0pi(df)
def sel_full(df):     return sel_0pi(df)     & c_muon(df)

CUT_STEPS = [
    ('Presel.',         sel_presel),
    ('FM',              sel_flash),
    ('Electron ID',     sel_shower),
    ('dE/dx',           sel_dedx),
    ('Angle, gap',      sel_anglegap),
    ('Proton ID',       sel_proton),
    ('$0\\pi$',          sel_0pi),
    ('LE-$\\mu$ veto',   sel_full),
]

sig = signal_mask(df)
print(f'True signal: {sig.sum():,}  |  Full selection: {sel_full(df).sum():,}')
print(f'Efficiency: {(sel_full(df)&sig).sum()/sig.sum():.3f}  |  Purity: {(sel_full(df)&sig).sum()/sel_full(df).sum():.3f}')

## 3. Efficiency curve (sanity check vs CAFAna)
---

In [ ]:
E_BINS = np.array([0.0,0.2,0.4,0.6,0.8,1.0,1.2,1.4,1.6,1.8,2.0,2.4,2.8,3.2,4.0])
E_CEN  = 0.5*(E_BINS[:-1]+E_BINS[1:])
E_WID  = np.diff(E_BINS)

fig, ax = plt.subplots(figsize=(10, 5.5), dpi=300)

denom, _ = np.histogram(df.loc[sig,'true_nu_energy'].dropna(), bins=E_BINS)

for (label, cut_fn), col in zip(CUT_STEPS, SEL_COLS):
    mask = cut_fn(df) & sig
    num, _ = np.histogram(df.loc[mask,'true_nu_energy'].dropna(), bins=E_BINS)
    eff  = np.where(denom>0, num/denom, np.nan)
    err  = np.where(denom>0, np.sqrt(eff*(1-eff)/np.where(denom>0,denom,1)), np.nan)
    tot  = mask.sum()/sig.sum()

    # shaded band instead of caps
    ax.fill_between(E_CEN, eff-err, eff+err,
                    color=col, alpha=0.15, step='mid', zorder=2)
    ax.errorbar(E_CEN, eff, xerr=E_WID/2,
                fmt='o', ms=4.5, lw=1.8, capsize=0, color=col,
                markeredgecolor='white', markeredgewidth=0.5,
                label=f'{label} ({100*tot:.0f}%)', zorder=5)

# truth backdrop on twin axis
ax2 = ax.twinx()
ax2.hist(df.loc[sig,'true_nu_energy'].dropna(), bins=E_BINS,
         color='#bdbdbd', alpha=0.22, zorder=0)
ax2.set_ylabel('Signal events', color='#999999', fontsize=11)
ax2.tick_params(axis='y', colors='#999999')
ax2.set_ylim(bottom=0)
ax2.grid(False)

ax.axhline(1.0, ls='--', lw=1.5, color='#555555', zorder=5)
ax.set_xlabel(r'$E_{\nu}$ [GeV]')
ax.set_ylabel('Selection efficiency')
ax.set_xlim(0, 4); ax.set_ylim(0, 1.35)
ax.legend(ncol=3, loc='upper right', fontsize=9)
ax.set_title('CC1e0pi selection efficiency (no trigger, Pandora-only)')
fig.tight_layout()
#savefig(fig, 'efficiency_python.png')
plt.show()

## 4. Cut flow table
---
Summary of the impact of each sequential cut on selection (statistics, efficiencies, purities). 

In [ ]:
rows = []
n_sig_tot = sig.sum()
for label, cut_fn in CUT_STEPS:
    sel   = cut_fn(df)
    n_sel = sel.sum()
    n_sig = (sel & sig).sum()
    n_bkg = n_sel - n_sig
    eff   = n_sig/n_sig_tot if n_sig_tot>0 else 0
    pur   = n_sig/n_sel     if n_sel>0     else 0
    punzi = eff/(1+np.sqrt(max(n_bkg,0)))
    rows.append({'Cut': label.replace('$','').replace('\\',''),
                 'N sel': f'{n_sel:,}', 'N sig': f'{n_sig:,}',
                 'Eff (%)': f'{100*eff:.1f}',
                 'Pur (%)': f'{100*pur:.1f}',
                 'Punzi FOM': f'{punzi:.5f}'})

cf = pd.DataFrame(rows)
print(cf.to_string(index=False))

# ── Visual cut flow bar chart ─────────────────────────────────────────────────
effs_cf = [float(r['Eff (%)']) for r in rows]
purs_cf = [float(r['Pur (%)']) for r in rows]
labels_cf = [r['Cut'] for r in rows]
x = np.arange(len(labels_cf))

fig, ax = plt.subplots(figsize=(11, 5), dpi=300)
ax.bar(x-0.2, effs_cf, 0.38, label='Efficiency (%)', color="powderblue",   alpha=0.85, zorder=3)
ax.bar(x+0.2, purs_cf, 0.38, label='Purity (%)',     color="black", alpha=0.85, zorder=3)

for i, (e, p) in enumerate(zip(effs_cf, purs_cf)):
    ax.text(i-0.2, e+0.5, f'{e:.0f}', ha='center', va='bottom', fontsize=8, color="powderblue")
    ax.text(i+0.2, p+0.5, f'{p:.0f}', ha='center', va='bottom', fontsize=8, color="black")

ax.set_xticks(x)
ax.set_xticklabels(labels_cf, rotation=25, ha='right', fontsize=10)
ax.set_ylabel('Percentage (%)')
ax.set_title('Cut flow: efficiency and purity at each step')
ax.legend()
fig.tight_layout()
#savefig(fig, 'cutflow_bar.png')
plt.show()

## 5. Electron KE threshold sweep
---

Sweep the true electron KE threshold (0.075--0.5 GeV) used in the signal definition.
The Pareto map shows the efficiency-purity tradeoff coloured by threshold value.
The crimson dot marks the baseline (0.2 GeV) and the gold star marks the point
maximising eff $\\times$ purity.

In [ ]:
KE_VALS   = np.array([0.075,0.100,0.125,0.150,0.175,0.200,
                       0.225,0.250,0.300,0.350,0.400,0.450,0.500])
BASE_KE   = 0.200
base_idx  = np.argmin(np.abs(KE_VALS-BASE_KE))

effs_ke, purs_ke, nsigs_ke = [], [], []
for ke in KE_VALS:
    sk   = signal_mask(df, ke)
    sel  = sel_full(df)
    n_s  = (sel & sk).sum()
    effs_ke.append(n_s/sk.sum()   if sk.sum()>0   else 0)
    purs_ke.append(n_s/sel.sum()  if sel.sum()>0  else 0)
    nsigs_ke.append(sk.sum())

effs_ke  = np.array(effs_ke)
purs_ke  = np.array(purs_ke)
nsigs_ke = np.array(nsigs_ke)
fom_ke   = effs_ke * purs_ke
opt_idx  = np.argmax(fom_ke)

# Binomial errors
eff_err = np.sqrt(effs_ke*(1-effs_ke)/np.maximum(nsigs_ke,1))
pur_err = np.sqrt(purs_ke*(1-purs_ke)/np.maximum(sel_full(df).sum(),1)) * np.ones_like(purs_ke)

fig = plt.figure(figsize=(17, 5), dpi=300)
gs  = fig.add_gridspec(1, 4, wspace=0.38)
axes = [fig.add_subplot(gs[i]) for i in range(4)]

for ax, vals, err, ylabel in [
        (axes[0], effs_ke,  eff_err, 'Efficiency'),
        (axes[1], purs_ke,  pur_err, 'Purity'),
        (axes[2], nsigs_ke, np.sqrt(nsigs_ke), 'N true signal events')]:
    band_plot(ax, KE_VALS, vals, err, BLUE, lw=2, ms=5)
    vline_base(ax, BASE_KE)
    vline_opt(ax, KE_VALS[opt_idx])
    ax.set_xlabel(r'True $e^{\pm}$ KE threshold [GeV]')
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=8)

# Pareto map
ax3 = axes[3]
sc  = ax3.scatter(effs_ke, purs_ke, c=KE_VALS, cmap='plasma',
                  s=55, zorder=5, edgecolors='white', linewidths=0.4)
ax3.plot(effs_ke, purs_ke, '-', color='#aaaaaa', lw=1, zorder=3)
plt.colorbar(sc, ax=ax3, label=r'KE threshold [GeV]')

# error ellipses at baseline and optimum
for idx, color, marker, ms in [(base_idx, CRIMSON, 'o', 120),
                                 (opt_idx,  GOLD,   '*', 160)]:
    ax3.scatter(effs_ke[idx], purs_ke[idx],
                s=ms, color=color, marker=marker, zorder=7,
                edgecolors='white', linewidths=0.8)
    ax3.errorbar(effs_ke[idx], purs_ke[idx],
                 xerr=eff_err[idx], yerr=pur_err[idx],
                 fmt='none', color=color, lw=1.5, capsize=4, zorder=6)

# dashed crosshairs at baseline
ax3.axhline(purs_ke[base_idx], ls='--', lw=0.9, color=CRIMSON, alpha=0.4)
ax3.axvline(effs_ke[base_idx], ls='--', lw=0.9, color=CRIMSON, alpha=0.4)

from matplotlib.lines import Line2D
ax3.legend(handles=[
    Line2D([0],[0],marker='o',color='w',markerfacecolor=CRIMSON,ms=9,
           label=f'Baseline ({BASE_KE} GeV)'),
    Line2D([0],[0],marker='*',color='w',markerfacecolor=GOLD,ms=11,
           label=f'Opt. eff*pur ({KE_VALS[opt_idx]:.3f} GeV)'),
], fontsize=8)
ax3.set_xlabel('Efficiency')
ax3.set_ylabel('Purity')
ax3.set_title('Efficiency-purity Pareto frontier')

fig.suptitle(r'True $e^{\pm}$ KE threshold sweep (no trigger, Pandora)',
             fontsize=13, y=1.02)
#savefig(fig, 'ke_threshold_sweep.png')
plt.show()

print(f'Baseline ({BASE_KE} GeV): eff={effs_ke[base_idx]:.3f}, pur={purs_ke[base_idx]:.3f}')
print(f'Optimum  ({KE_VALS[opt_idx]:.3f} GeV): eff={effs_ke[opt_idx]:.3f}, pur={purs_ke[opt_idx]:.3f}')

## 6. Individual cut sweeps with Punzi FOM
---

Each cut is swept while all others are held at baseline.
**Punzi FOM** $= \varepsilon / (1 + \sqrt{N_{B}})$ is the standard figure of
merit for optimising a cut targeting 90% CL sensitivity in a counting experiment.
The gold dotted line marks the FOM-optimal threshold for each cut.

In [ ]:
def run_sweep(df, sig, vals, full_fn):
    n_tot = sig.sum()
    effs, purs, foms = [], [], []
    for v in vals:
        sel   = full_fn(df, v)
        n_sig = (sel & sig).sum()
        n_sel = sel.sum()
        n_bkg = n_sel - n_sig
        eff   = n_sig/n_tot if n_tot>0 else 0
        pur   = n_sig/n_sel if n_sel>0 else 0
        effs.append(eff)
        purs.append(pur)
        foms.append(eff/(1+np.sqrt(max(n_bkg,0))))
    e, p, f = np.array(effs), np.array(purs), np.array(foms)
    eff_err = np.sqrt(e*(1-e)/max(n_tot,1))
    pur_err = np.sqrt(p*(1-p)/np.maximum(np.array([sel_full(df).sum()]*len(vals)),1))
    return e, p, f, eff_err, pur_err

# full selection with one free parameter
def fs_e(df,v):   return sel_flash(df)    &c_shower(df,v)&c_dedx(df) &c_angle(df)&c_gap(df)&c_proton(df)&c_0pi(df)&c_muon(df)
def fs_dx(df,v):  return sel_shower(df)   &c_dedx(df,v)  &c_angle(df)&c_gap(df)  &c_proton(df)&c_0pi(df)&c_muon(df)
def fs_ang(df,v): return sel_dedx(df)     &c_angle(df,v) &c_gap(df)  &c_proton(df)&c_0pi(df)&c_muon(df)
def fs_gap(df,v): return sel_dedx(df)     &c_angle(df)   &c_gap(df,v)&c_proton(df)&c_0pi(df)&c_muon(df)
def fs_mu(df,v):  return sel_0pi(df)      &c_muon(df,v)

sweeps = [
    ('Shower energy min [GeV]',            np.linspace(0.05,0.5,35),  SHOWER_ENERGY_MIN,  fs_e),
    ('Shower dE/dx max [MeV/cm]',          np.linspace(0.8, 9.0,35),  SHOWER_DEDX_MAX,    fs_dx),
    ('Open angle max [deg]',               np.linspace(2.0,40.0,35),  SHOWER_ANGLE_MAX,   fs_ang),
    ('Conv. gap max [cm]',                 np.linspace(0.5,15.0,35),  SHOWER_CONVGAP_MAX, fs_gap),
    (r'$\chi^{2}_{\mu}$ veto max',          np.linspace(10.,100.,35), MUON_CHI2_MU_MAX,   fs_mu),
]

fig, axes = plt.subplots(3, len(sweeps), figsize=(20,11),
                          gridspec_kw={'hspace':0.50,'wspace':0.38}, dpi=300)

for col, (xlabel, vals, base, sfn) in enumerate(sweeps):
    effs, purs, foms, eff_err, pur_err = run_sweep(df, sig, vals, sfn)
    opt_v = vals[np.argmax(foms)]
    fom_err = np.abs(foms) * np.sqrt((eff_err/np.where(effs>0,effs,1))**2)

    for row, (y, err, ylabel) in enumerate([
            (effs, eff_err, 'Efficiency'),
            (purs, pur_err, 'Purity'),
            (foms, fom_err, 'Punzi FOM')]):
        ax = axes[row, col]
        band_plot(ax, vals, y, err, BLUE, lw=2, ms=0, fmt='')
        ax.plot(vals, y, '-', color=BLUE, lw=2, zorder=4)
        vline_base(ax, base)
        vline_opt(ax, opt_v)
        ax.set_xlabel(xlabel, fontsize=9)
        ax.set_ylabel(ylabel, fontsize=10)
        if row==0:
            ax.set_title(xlabel.split('[')[0].strip(), fontsize=9)
        if col==0 or row==2:
            ax.legend(fontsize=7)

fig.suptitle('Individual cut sweeps -- efficiency, purity, Punzi FOM\n'
             '(no trigger, Pandora-only, all others at baseline)',
             fontsize=12, y=1.01)
#savefig(fig, 'cut_sweeps.png')
plt.show()

## 7. Cut ordering study
---
Does the order in which various cuts are applied impact efficiencies and purities?

In [ ]:
ATOMIC = {
    'Presel.':   c_presel,  'FM':        c_flash,
    'Shower E':  c_shower,  'dE/dx':     c_dedx,
    'Angle':     c_angle,   'Gap':       c_gap,
    'Proton':    c_proton,  '$0\\pi$':    c_0pi,
    'Muon veto': c_muon,
}

def apply_ordering(df, sig, order):
    effs, purs = [], []
    mask = pd.Series(True, index=df.index)
    n_tot = sig.sum()
    for name in order:
        mask  = mask & ATOMIC[name](df)
        n_sig = (mask & sig).sum()
        n_sel = mask.sum()
        effs.append(n_sig/n_tot  if n_tot>0  else 0)
        purs.append(n_sig/n_sel  if n_sel>0  else 0)
    return np.array(effs), np.array(purs)

ORDER_STD  = ['Presel.','FM','Shower E','dE/dx','Angle','Gap','Proton','$0\\pi$','Muon veto']
ORDER_DEDX = ['Presel.','FM','dE/dx','Shower E','Angle','Gap','Proton','$0\\pi$','Muon veto']
ORDER_TOPO = ['Presel.','FM','Proton','$0\\pi$','Shower E','dE/dx','Angle','Gap','Muon veto']

ords = [
    ('Standard',       ORDER_STD,  '#4257f5'),
    ('dE/dx first',    ORDER_DEDX, '#f39c12'),
    ('Topology first', ORDER_TOPO, '#3f8f3f'),
]

fig, (ax_e, ax_p) = plt.subplots(1,2, figsize=(14,5.5), sharey=False)
x = np.arange(len(ORDER_STD))

for name, order, col in ords:
    effs, purs = apply_ordering(df, sig, order)
    eff_err = np.sqrt(effs*(1-effs)/max(sig.sum(),1))
    pur_err = np.sqrt(purs*(1-purs)/np.maximum(
        [sum(pd.Series(True,index=df.index)
             &pd.concat([ATOMIC[k](df) for k in order[:i+1]],axis=1).all(axis=1))
         for i in range(len(order))], 1))

    for ax, vals, err in [(ax_e, effs, eff_err),(ax_p, purs, pur_err)]:
        ax.fill_between(x, vals-err, vals+err, color=col, alpha=0.15, zorder=2)
        ax.plot(x, vals, 'o-', color=col, lw=2, ms=5.5, label=name,
                markeredgecolor='white', markeredgewidth=0.5, zorder=4)

xlabs = [s.replace('$','').replace('\\','') for s in ORDER_STD]
for ax, ylabel, title in [(ax_e,'Efficiency','Efficiency vs cut ordering'),
                           (ax_p,'Purity',    'Purity vs cut ordering')]:
    ax.set_xticks(list(x))
    ax.set_xticklabels(xlabs, rotation=30, ha='right', fontsize=9)
    ax.set_ylabel(ylabel)
    ax.set_ylim(0,1.05)
    ax.set_title(title)
    ax.legend()

fig.suptitle('Cut ordering study (no trigger, Pandora)', y=1.02)
fig.tight_layout()
#savefig(fig, 'cut_ordering.png')
plt.show()

## 8. 2D efficiency and purity maps (smoothed)
---

Examples --> trade-off in threshold placement of various cut variables can be visualised this way. 

In [ ]:
def smooth_nan(a, sigma=1.2):
    """NaN-aware Gaussian smoothing."""
    filled = np.where(np.isnan(a), 0., a)
    weight = np.where(np.isnan(a), 0., 1.)
    sf = gaussian_filter(filled, sigma)
    sw = gaussian_filter(weight, sigma)
    return np.where(sw > 0.1, sf/sw, np.nan)

def plot_2d(df, sig, vx, vy, lx, ly, bx, by, sel, title, fname,
            min_count=3, sigma=1.2):
    ok = df[vx].notna() & df[vy].notna()
    def h2(mask):
        return np.histogram2d(df.loc[ok&mask,vx], df.loc[ok&mask,vy],
                              bins=[bx,by])[0]
    den = h2(sig)
    num = h2(sig&sel)
    tot = h2(sel)

    eff = np.where(den>min_count, num/den, np.nan)
    pur = np.where(tot>min_count, num/tot, np.nan)
    eff_s = smooth_nan(eff, sigma)
    pur_s = smooth_nan(pur, sigma)

    fig, axes = plt.subplots(1,2, figsize=(14,5.5))
    for ax, data, clabel in zip(axes,[eff_s,pur_s],['Efficiency','Purity']):
        im = ax.pcolormesh(bx, by, data.T, cmap=MANAGUA, vmin=0, vmax=1)
        cb = plt.colorbar(im, ax=ax, label=clabel)
        cb.ax.tick_params(labelsize=10)
        ax.set_xlabel(lx)
        ax.set_ylabel(ly)
        ax.set_title(f'{clabel} -- {title}')

    fig.tight_layout()
    #savefig(fig, fname)
    plt.show()

sel = sel_full(df)

plot_2d(df, sig,
    'shower_energy','shower_avail_dedx',
    'Shower energy [GeV]','Shower dE/dx [MeV/cm]',
    np.linspace(0,2.5,30), np.linspace(0,10,30),
    sel, 'shower energy vs dE/dx', 'map_energy_dedx.png')

plot_2d(df, sig,
    'shower_energy','shower_open_angle',
    'Shower energy [GeV]','Open angle [deg]',
    np.linspace(0,2.5,30), np.linspace(0,40,30),
    sel, 'shower energy vs open angle', 'map_energy_angle.png')

# Proton momentum vs leading proton chi2
plot_2d(df, sig,
    'lead_proton_momentum','lead_proton_chi2_proton',
    'Leading proton momentum [GeV/c]',r'Proton $\chi^{2}_{p}$',
    np.linspace(0,1.5,28), np.linspace(0,200,28),
    sel, 'proton momentum vs chi2', 'map_proton_chi2.png')

## 9. Neutrino energy resolution and regression
---

Linear fit $E_{reco} = m \cdot E_{true} + c$ for selected signal events.
Slope < 1 indicates systematic underestimation (expected: missing neutron energy,
sub-threshold protons, calorimetric inefficiency).

Resolution $\sigma[(E_{reco}-E_{true})/E_{true}]$ is fitted with a linear
model $\sigma = a + b\cdot E_{true}$ to characterise the energy dependence.

In [ ]:
ok = (sel_full(df) & sig &
      df['reco_nu_energy'].notna() & df['true_nu_energy'].notna() &
      (df['reco_nu_energy']>0))
Et = df.loc[ok,'true_nu_energy'].values
Er = df.loc[ok,'reco_nu_energy'].values
res = (Er - Et)/Et

# linear fit with uncertainties
m, c, r, p, se = linregress(Et, Er)
Ef  = np.linspace(0.1, 3.5, 300)
Ep  = m*Ef + c
n, Em = len(Et), Et.mean()
sxx  = ((Et-Em)**2).sum()
band = se * np.sqrt(1/n + (Ef-Em)**2/sxx)   # prediction band

# binned residuals
RB  = np.linspace(0.2, 3.2, 10)
RC  = 0.5*(RB[:-1]+RB[1:])
r_mu, r_sig, r_err = [], [], []
for lo,hi in zip(RB[:-1],RB[1:]):
    rb = res[(Et>=lo)&(Et<hi)]
    if len(rb)>5:
        r_mu.append(rb.mean()); r_sig.append(rb.std())
        r_err.append(rb.std()/np.sqrt(len(rb)))
    else:
        r_mu.append(np.nan); r_sig.append(np.nan); r_err.append(np.nan)
r_mu  = np.array(r_mu)
r_sig = np.array(r_sig)
r_err = np.array(r_err)

# resolution fit sigma = a + b*E
ok_b = ~np.isnan(r_sig)
try:
    (a_res, b_res), cov_res = curve_fit(lambda x,a,b: a+b*x, RC[ok_b], r_sig[ok_b])
    perr_res = np.sqrt(np.diag(cov_res))
    fit_ok = True
except:
    fit_ok = False

# ── Figure: 2x2 panels ────────────────────────────────────────────────────────
fig = plt.figure(figsize=(14,10))
gs  = fig.add_gridspec(2, 2, hspace=0.40, wspace=0.35)
ax_main = fig.add_subplot(gs[0,:])
ax_res  = fig.add_subplot(gs[1,0])
ax_sig  = fig.add_subplot(gs[1,1])

# ── Main: 2D + fit ────────────────────────────────────────────────────────────
h,xb,yb,im = ax_main.hist2d(Et, Er, bins=32, range=[[0,3],[0,3]], cmap=MANAGUA)
plt.colorbar(im, ax=ax_main, label='Events')

ax_main.plot(Ef, Ef, 'w--', lw=1.8, label='Perfect reconstruction', zorder=5)
ax_main.plot(Ef, Ep, color='#ff6b6b', lw=2.5, zorder=6,
             label=fr'Fit: $E_{{reco}} = {m:.3f}\,E_{{true}} {c:+.3f}$ ($R^2={r**2:.3f}$)')
ax_main.fill_between(Ef, Ep-band, Ep+band,
                      color='#ff6b6b', alpha=0.22, zorder=4,
                      label=r'$1\sigma$ fit uncertainty')

ax_main.set_xlabel(r'True $E_{\nu}$ [GeV]')
ax_main.set_ylabel(r'Reco $E_{\nu}$ [GeV]')
ax_main.set_title(r'Neutrino energy: reco vs true (selected signal, no trigger, Pandora)')
ax_main.legend(fontsize=9, loc='upper left')
ax_main.set_xlim(0,3); ax_main.set_ylim(0,3)

# ── Bottom left: mean residual ────────────────────────────────────────────────
ax_res.axhline(0, ls='--', lw=1.5, color='#555555', zorder=5)
ax_res.fill_between(RC, r_mu-r_sig, r_mu+r_sig,
                     color=BLUE, alpha=0.15, step='mid', zorder=2,
                     label=r'$\pm 1\sigma$ spread')
ax_res.errorbar(RC, r_mu, yerr=r_err,
                fmt='o', ms=6, lw=1.8, capsize=4, color=BLUE,
                markeredgecolor='white', markeredgewidth=0.5,
                label='Mean residual', zorder=5)
ax_res.set_xlabel(r'True $E_{\nu}$ [GeV]')
ax_res.set_ylabel(r'$(E_{reco}-E_{true})/E_{true}$')
ax_res.set_title('Mean fractional residual vs energy')
ax_res.legend(fontsize=9)

# ── Bottom right: resolution (sigma) ─────────────────────────────────────────
if fit_ok:
    ax_sig.plot(Ef, a_res+b_res*Ef, color='#ff6b6b', lw=2.2, zorder=4,
                label=fr'Fit: $\sigma={a_res:.3f}\pm{perr_res[0]:.3f}$ '
                      fr'$+({b_res:.3f}\pm{perr_res[1]:.3f})\,E_{{true}}$')
    ax_sig.fill_between(Ef,
        (a_res-perr_res[0])+(b_res-perr_res[1])*Ef,
        (a_res+perr_res[0])+(b_res+perr_res[1])*Ef,
        color='#ff6b6b', alpha=0.18, zorder=3)
    print(f'Resolution: sigma = {a_res:.4f}(+-{perr_res[0]:.4f}) + {b_res:.4f}(+-{perr_res[1]:.4f}) * E_true')

ax_sig.fill_between(RC[ok_b], (r_sig-r_err)[ok_b], (r_sig+r_err)[ok_b],
                    color=BLUE, alpha=0.18, step='mid', zorder=2)
ax_sig.plot(RC[ok_b], r_sig[ok_b], 'o', color=BLUE, ms=6, zorder=5,
            markeredgecolor='white', markeredgewidth=0.5, label='Resolution')
ax_sig.set_xlabel(r'True $E_{\nu}$ [GeV]')
ax_sig.set_ylabel(r'$\sigma[(E_{reco}-E_{true})/E_{true}]$')
ax_sig.set_title('Energy resolution (RMS) vs energy')
ax_sig.set_ylim(bottom=0)
ax_sig.legend(fontsize=8)

print(f'Linear fit: E_reco = {m:.4f} * E_true {c:+.4f}  (R^2={r**2:.4f})')
print(f'Slope < 1 ({m:.4f}) => systematic underestimation of E_nu')

#savefig(fig, 'energy_resolution.png')
plt.show()

## Energy Resolution: Fitting Method Notes
---
### The problem
Each vertical slice of the reco vs true E_nu 2D map has an asymmetric
distribution — a sharp rising edge near the peak and a long tail dragging to lower reco energies. This tail comes from events where significant energy is lost through mechanisms that are not visible: neutrons, sub-threshold protons, and hadronic activity below the detection threshold. A naive mean of this distribution is pulled far below the physical peak, making the fit appear worse than it is. The goal is to extract the **most probable value (MPV)** — the peak of the distribution — not the mean.


In [ ]:
from scipy.stats import norm as sp_norm

# redefine signal mask inline to avoid any variable clash
sig_mask = (df['is_signal']==1) & (df['true_electron_ke'].fillna(0) > 0.2)

ok = (np.array(sel_full(df), dtype=bool) &
      np.array(sig_mask, dtype=bool) &
      np.array(df['reco_nu_energy'].notna(), dtype=bool) &
      np.array(df['true_nu_energy'].notna(), dtype=bool) &
      np.array(df['reco_nu_energy'].values > 0, dtype=bool))

Et  = df['true_nu_energy'].values[ok]
Er  = df['reco_nu_energy'].values[ok]
res = (Er - Et) / Et

print(f'N events: {ok.sum()}')
print(f'Mean residual: {res.mean():.3f}')
print(f'Et range: {Et.min():.2f} - {Et.max():.2f} GeV')
print(f'Er range: {Er.min():.2f} - {Er.max():.2f} GeV')

### Method 1: Symmetric Gaussian 
A standard Gaussian fit to each E_true slice finds the mean and width.
Since the distribution is asymmetric, the mean sits significantly below
the peak — the long low-energy tail pulls it down. The resulting fit line lies well below the data ridge, and residuals are systematically negative with no way to be symmetric around zero. The Gaussian is the wrong functional form for this distribution.

In [ ]:
# redefine signal mask inline to avoid any variable clash
sig_mask = (df['is_signal']==1) & (df['true_electron_ke'].fillna(0) > 0.2)

ok = (np.array(sel_full(df), dtype=bool) &
      np.array(sig_mask, dtype=bool) &
      np.array(df['reco_nu_energy'].notna(), dtype=bool) &
      np.array(df['true_nu_energy'].notna(), dtype=bool) &
      np.array(df['reco_nu_energy'].values > 0, dtype=bool))

Et  = df['true_nu_energy'].values[ok]
Er  = df['reco_nu_energy'].values[ok]
res = (Er - Et) / Et

print(f'N events: {ok.sum()}')
print(f'Mean residual: {res.mean():.3f}')

# Gaussian peak per E_true bin
FIT_LO, FIT_HI = 0.3, 2.2
E_TRUE_BINS = np.linspace(FIT_LO, FIT_HI, 12)
E_TRUE_CEN  = 0.5*(E_TRUE_BINS[:-1]+E_TRUE_BINS[1:])

peak_reco, peak_err = [], []
for lo, hi in zip(E_TRUE_BINS[:-1], E_TRUE_BINS[1:]):
    er_bin = Er[(Et>=lo) & (Et<hi)]
    if len(er_bin) > 10:
        try:
            mu, sg = sp_norm.fit(er_bin)   # sg not sig!
            peak_reco.append(mu)
            peak_err.append(sg/np.sqrt(len(er_bin)))
        except:
            peak_reco.append(np.nan); peak_err.append(np.nan)
    else:
        peak_reco.append(np.nan); peak_err.append(np.nan)

peak_reco = np.array(peak_reco)
peak_err  = np.array(peak_err)
ok_p      = ~np.isnan(peak_reco)

# linear fit through Gaussian peaks
m, c, r_val, _, se = linregress(E_TRUE_CEN[ok_p], peak_reco[ok_p])
Ef    = np.linspace(FIT_LO, FIT_HI, 300)
Ep    = m*Ef + c
n_fit = ok_p.sum()
Em    = E_TRUE_CEN[ok_p].mean()
sxx   = ((E_TRUE_CEN[ok_p]-Em)**2).sum()
band  = se * np.sqrt(1/n_fit + (Ef-Em)**2/sxx)

print(f'Ridge fit (Gaussian peaks, {FIT_LO}-{FIT_HI} GeV):')
print(f'  slope     = {m:.4f}')
print(f'  intercept = {c:.4f} GeV')
print(f'  R^2       = {r_val**2:.4f}')

# Binned residuals 
RB = np.linspace(0.2, 3.2, 10)
RC = 0.5*(RB[:-1]+RB[1:])
r_mu, r_sg, r_err = [], [], []
for lo, hi in zip(RB[:-1], RB[1:]):
    rb = res[(Et>=lo) & (Et<hi)]
    if len(rb) > 5:
        r_mu.append(rb.mean())
        r_sg.append(rb.std())
        r_err.append(rb.std()/np.sqrt(len(rb)))
    else:
        r_mu.append(np.nan); r_sg.append(np.nan); r_err.append(np.nan)

r_mu  = np.array(r_mu)
r_sg  = np.array(r_sg)
r_err = np.array(r_err)

# Resolution fit sigma = a + b*E 
ok_b = ~np.isnan(r_sg)
try:
    (a_res,b_res), cov_res = curve_fit(lambda x,a,b: a+b*x, RC[ok_b], r_sg[ok_b])
    perr_res = np.sqrt(np.diag(cov_res))
    fit_ok = True
    print(f'  Resolution: sigma = {a_res:.3f}+-{perr_res[0]:.3f} + ({b_res:.3f}+-{perr_res[1]:.3f})*E_true')
except:
    fit_ok = False

# Figure 
fig = plt.figure(figsize=(14,10))
gs  = fig.add_gridspec(2, 2, hspace=0.42, wspace=0.35)
ax_main = fig.add_subplot(gs[0,:])
ax_res  = fig.add_subplot(gs[1,0])
ax_sg   = fig.add_subplot(gs[1,1])

# Main: 2D map + ridge fit 
h,xb,yb,im = ax_main.hist2d(Et, Er, bins=32, range=[[0,3],[0,3]], cmap=MANAGUA)
plt.colorbar(im, ax=ax_main, label='Events')

ax_main.plot([0,3],[0,3], 'w--', lw=1.8, label='Perfect reconstruction', zorder=5)
ax_main.errorbar(E_TRUE_CEN[ok_p], peak_reco[ok_p], yerr=peak_err[ok_p],
                 fmt='o', ms=5, color='white', markeredgecolor='#ff6b6b',
                 markeredgewidth=1.5, capsize=3, lw=1.2, zorder=7,
                 label='Gaussian peak per bin')
ax_main.plot(Ef, Ep, color='#ff6b6b', lw=2.5, zorder=6,
             label=fr'Ridge fit: $E_{{reco}}={m:.3f}\,E_{{true}}{c:+.3f}$ ($R^2={r_val**2:.3f}$)')
ax_main.fill_between(Ef, Ep-band, Ep+band,
                     color='#ff6b6b', alpha=0.22, zorder=4, label=r'$1\sigma$ fit band')
ax_main.axvline(FIT_LO, ls=':', lw=1, color='#ff6b6b', alpha=0.4)
ax_main.axvline(FIT_HI, ls=':', lw=1, color='#ff6b6b', alpha=0.4)

ax_main.set_xlabel(r'True $E_{\nu}$ [GeV]')
ax_main.set_ylabel(r'Reco $E_{\nu}$ [GeV]')
ax_main.set_title(r'Neutrino energy: reco vs true (selected signal, no trigger, Pandora)')
ax_main.legend(fontsize=9, loc='upper left')
ax_main.set_xlim(0,3); ax_main.set_ylim(0,3)

# Bottom left: mean residual
ax_res.axhline(0, ls='--', lw=1.5, color='#555555', zorder=5)
ax_res.fill_between(RC, r_mu-r_sg, r_mu+r_sg,
                    color=BLUE, alpha=0.15, step='mid', zorder=2,
                    label=r'$\pm 1\sigma$ spread')
ax_res.errorbar(RC, r_mu, yerr=r_err,
                fmt='o', ms=6, lw=1.8, capsize=4, color=BLUE,
                markeredgecolor='white', markeredgewidth=0.5,
                label='Mean residual', zorder=5)
ax_res.set_xlabel(r'True $E_{\nu}$ [GeV]')
ax_res.set_ylabel(r'$(E_{reco}-E_{true})/E_{true}$')
ax_res.set_title('Mean fractional residual vs energy')
ax_res.legend(fontsize=9)

# Bottom right: resolution 
Ef_res = np.linspace(0.2, 3.2, 200)
if fit_ok:
    ax_sg.plot(Ef_res, a_res+b_res*Ef_res, color='#ff6b6b', lw=2.2, zorder=4,
               label=fr'Fit: $\sigma={a_res:.3f}\pm{perr_res[0]:.3f}'
                     fr'+({b_res:.3f}\pm{perr_res[1]:.3f})\,E_{{true}}$')
    ax_sg.fill_between(Ef_res,
        np.maximum(0,(a_res-perr_res[0])+(b_res-perr_res[1])*Ef_res),
        (a_res+perr_res[0])+(b_res+perr_res[1])*Ef_res,
        color='#ff6b6b', alpha=0.18, zorder=3)

ax_sg.fill_between(RC[ok_b], np.maximum(0,(r_sg-r_err)[ok_b]), (r_sg+r_err)[ok_b],
                   color=BLUE, alpha=0.18, step='mid', zorder=2)
ax_sg.plot(RC[ok_b], r_sg[ok_b], 'o', color=BLUE, ms=6, zorder=5,
           markeredgecolor='white', markeredgewidth=0.5, label='Resolution')
ax_sg.set_xlabel(r'True $E_{\nu}$ [GeV]')
ax_sg.set_ylabel(r'$\sigma[(E_{reco}-E_{true})/E_{true}]$')
ax_sg.set_title('Energy resolution (RMS) vs energy')
ax_sg.set_ylim(bottom=0)
ax_sg.legend(fontsize=8)

#savefig(fig, 'energy_resolution.png')
plt.show()

### Method 2: KDE peak (kernel density estimate)
Instead of fitting a parametric function, a KDE smooths the data with a
Gaussian kernel and finds the x-value of the maximum density. This is
non-parametric and makes no assumption about the shape. It works better
than a symmetric Gaussian because it can find the true mode of an
asymmetric distribution. However, it is sensitive to the bandwidth
parameter: too wide and the peak is smeared, too narrow and statistical
noise dominates. With ~30 events per bin the KDE peak estimate is noisy
and jumps around between bins, giving inconsistent MPVs and residuals
that do not sit stably around zero.

In [ ]:
# Signal mask (inline to avoid variable clash) 
sig_mask = (df['is_signal']==1) & (df['true_electron_ke'].fillna(0) > 0.2)

ok = (np.array(sel_full(df), dtype=bool) &
      np.array(sig_mask, dtype=bool) &
      np.array(df['reco_nu_energy'].notna(), dtype=bool) &
      np.array(df['true_nu_energy'].notna(), dtype=bool) &
      np.array(df['reco_nu_energy'].values > 0, dtype=bool))

Et  = df['true_nu_energy'].values[ok]
Er  = df['reco_nu_energy'].values[ok]

print(f'N selected signal: {ok.sum()}')

# E_true bins for slice fits 
FIT_LO, FIT_HI = 0.3, 2.2
E_TRUE_BINS = np.linspace(FIT_LO, FIT_HI, 12)
E_TRUE_CEN  = 0.5*(E_TRUE_BINS[:-1]+E_TRUE_BINS[1:])
N_BINS      = len(E_TRUE_CEN)
N_RECO_BINS = 18

# Per-slice MPV (windowed histogram mode) 
peak_reco, peak_err = [], []
slice_data   = []
slice_counts = []
slice_edges  = []

from scipy.stats import gaussian_kde

# replace the inner loop with:
for lo, hi in zip(E_TRUE_BINS[:-1], E_TRUE_BINS[1:]):
    er_bin = Er[(Et>=lo) & (Et<hi)]
    slice_data.append(er_bin)
    if len(er_bin) > 8:
        expected = 0.5*(lo+hi)
        win_lo   = max(0,   expected - 0.6)
        win_hi   = min(3.5, expected + 0.6)
        
        # KDE peak finder
        er_win = er_bin[(er_bin>=win_lo) & (er_bin<=win_hi)]
        if len(er_win) > 5:
            kde      = gaussian_kde(er_win, bw_method=0.3)
            x_scan   = np.linspace(win_lo, win_hi, 200)
            mpv      = x_scan[np.argmax(kde(x_scan))]
            # bootstrap uncertainty on MPV
            boots = [x_scan[np.argmax(gaussian_kde(
                         er_win[np.random.choice(len(er_win), len(er_win))],
                         bw_method=0.3)(x_scan))]
                     for _ in range(50)]
            mpv_err = np.std(boots)
        else:
            mpv, mpv_err = np.nan, np.nan
            
        # still store histogram for the plot
        counts, edges = np.histogram(er_bin, bins=N_RECO_BINS,
                                     range=(win_lo, win_hi))
        slice_counts.append(counts); slice_edges.append(edges)
        peak_reco.append(mpv); peak_err.append(mpv_err)
    else:
        slice_counts.append(None); slice_edges.append(None)
        peak_reco.append(np.nan); peak_err.append(np.nan)

peak_reco = np.array(peak_reco)
peak_err  = np.array(peak_err)
ok_p      = ~np.isnan(peak_reco)

# Weighted linear fit through MPV peaks 
weights  = 1.0 / np.where(peak_err[ok_p]>0, peak_err[ok_p]**2, 1e6)
weights /= weights.sum()

coeffs, cov_p = np.polyfit(E_TRUE_CEN[ok_p], peak_reco[ok_p], 1,
                            w=weights, cov=True)
m, c   = coeffs
m_err  = np.sqrt(cov_p[0,0])
c_err  = np.sqrt(cov_p[1,1])

y_pred   = m*E_TRUE_CEN[ok_p] + c
ss_res   = np.sum(weights*(peak_reco[ok_p]-y_pred)**2)
ss_tot   = np.sum(weights*(peak_reco[ok_p] -
                            np.average(peak_reco[ok_p],weights=weights))**2)
r_val_sq = 1 - ss_res/ss_tot

Ef   = np.linspace(FIT_LO, FIT_HI, 300)
Ep   = m*Ef + c
band = np.sqrt((Ef*m_err)**2 + c_err**2)

print(f'Weighted MPV fit: E_reco = {m:.4f}+-{m_err:.4f} * E_true {c:+.4f}+-{c_err:.4f}')
print(f'Weighted R^2 = {r_val_sq:.4f}')

#  MPV-based residuals (consistent with ridge fit) 
r_mu  = np.where(ok_p, (peak_reco - E_TRUE_CEN) / E_TRUE_CEN, np.nan)
r_err = np.where(ok_p, peak_err / E_TRUE_CEN, np.nan)

# spread = RMS of (Er - ridge_predicted)/Et in each bin
r_sg = []
for i, (lo, hi) in enumerate(zip(E_TRUE_BINS[:-1], E_TRUE_BINS[1:])):
    mask_bin = (Et>=lo) & (Et<hi)
    er_bin   = Er[mask_bin]
    et_bin   = Et[mask_bin]
    if ok_p[i] and len(er_bin) > 5:
        predicted = m * E_TRUE_CEN[i] + c
        spread    = (er_bin - predicted) / np.where(et_bin>0, et_bin, 1)
        r_sg.append(np.std(spread))
    else:
        r_sg.append(np.nan)
r_sg = np.array(r_sg)

# Resolution fit sigma = a + b*E 
ok_b = ok_p & ~np.isnan(r_sg)
try:
    (a_res,b_res), cov_res = curve_fit(lambda x,a,b: a+b*x,
                                        E_TRUE_CEN[ok_b], r_sg[ok_b])
    perr_res = np.sqrt(np.diag(cov_res))
    fit_ok   = True
    print(f'Resolution: sigma = {a_res:.3f}+-{perr_res[0]:.3f} + ({b_res:.3f}+-{perr_res[1]:.3f})*E_true')
except:
    fit_ok = False

# =============================================================================
# PLOT 1: per-slice diagnostic
# =============================================================================
ncols = 4
nrows = int(np.ceil(N_BINS / ncols))
fig_s, axes_s = plt.subplots(nrows, ncols, figsize=(14, 3.2*nrows),
                              gridspec_kw={'hspace':0.55,'wspace':0.35})
axes_s = axes_s.flatten()

for i, (lo, hi) in enumerate(zip(E_TRUE_BINS[:-1], E_TRUE_BINS[1:])):
    ax    = axes_s[i]
    er_b  = slice_data[i]
    label = fr'$E_{{true}}\in[{lo:.2f},{hi:.2f}]$ GeV'

    if slice_counts[i] is not None and len(er_b) > 8:
        counts = slice_counts[i]
        edges  = slice_edges[i]
        cens   = 0.5*(edges[:-1]+edges[1:])
        width  = edges[1]-edges[0]
        errs   = np.sqrt(np.maximum(counts, 1))

        ax.bar(cens, counts, width=width*0.85,
               color=BLUE, alpha=0.55, zorder=2)
        ax.errorbar(cens, counts, yerr=errs, fmt='none',
                    color=BLUE, lw=1.2, capsize=2, zorder=3)

        if ok_p[i]:
            mpv = peak_reco[i]
            ax.axvline(mpv, color=CRIMSON, lw=2, ls='--', zorder=4,
                       label=f'MPV={mpv:.2f} GeV')
            ax.axvspan(mpv-peak_err[i], mpv+peak_err[i],
                       color=CRIMSON, alpha=0.15, zorder=3)
            # also show ridge prediction
            ridge_pred = m*E_TRUE_CEN[i] + c
            ax.axvline(ridge_pred, color=GOLD, lw=1.5, ls=':', zorder=4,
                       label=f'Ridge={ridge_pred:.2f} GeV')
            ax.legend(fontsize=7, loc='upper right')

        ax.text(0.04, 0.96, f'N={len(er_b)}', transform=ax.transAxes,
                fontsize=7, va='top', color='#444444')
    else:
        ax.text(0.5, 0.5, f'{label}\nInsufficient stats\n(N={len(er_b)})',
                ha='center', va='center', transform=ax.transAxes,
                fontsize=8, color='#888888')

    ax.set_title(label, fontsize=8)
    ax.set_xlabel(r'Reco $E_{\nu}$ [GeV]', fontsize=8)
    ax.set_ylabel('Events', fontsize=8)
    ax.tick_params(labelsize=7)

for j in range(i+1, len(axes_s)):
    axes_s[j].set_visible(False)

fig_s.suptitle(
    r'Reco $E_{\nu}$ distributions per true $E_{\nu}$ slice'
    '\n(crimson = MPV, gold = ridge prediction)',
    fontsize=11)
savefig(fig_s, 'energy_slices.png')
plt.show()

# =============================================================================
# PLOT 2: resolution figure
# =============================================================================
fig = plt.figure(figsize=(14,10))
gs  = fig.add_gridspec(2, 2, hspace=0.42, wspace=0.35)
ax_main = fig.add_subplot(gs[0,:])
ax_res  = fig.add_subplot(gs[1,0])
ax_sg   = fig.add_subplot(gs[1,1])

# Main panel 
h,xb,yb,im = ax_main.hist2d(Et, Er, bins=32, range=[[0,3],[0,3]], cmap=MANAGUA)
plt.colorbar(im, ax=ax_main, label='Events')

ax_main.plot([0,3],[0,3], 'w--', lw=1.8, label='Perfect reconstruction', zorder=5)
ax_main.errorbar(E_TRUE_CEN[ok_p], peak_reco[ok_p], yerr=peak_err[ok_p],
                 fmt='o', ms=5.5, color='white', markeredgecolor=CRIMSON,
                 markeredgewidth=1.8, capsize=3, lw=1.4, zorder=7,
                 label='MPV per slice')
ax_main.plot(Ef, Ep, color=CRIMSON, lw=2.5, zorder=6,
             label=fr'Weighted MPV fit: '
                   fr'$E_{{reco}}={m:.3f}\pm{m_err:.3f}\,E_{{true}}{c:+.3f}\pm{c_err:.3f}$'
                   fr'  ($R^2={r_val_sq:.3f}$)')
ax_main.fill_between(Ef, Ep-band, Ep+band, color=CRIMSON, alpha=0.20,
                     zorder=4, label=r'$1\sigma$ fit uncertainty')
ax_main.axvline(FIT_LO, ls=':', lw=1, color=CRIMSON, alpha=0.4)
ax_main.axvline(FIT_HI, ls=':', lw=1, color=CRIMSON, alpha=0.4)
ax_main.set_xlabel(r'True $E_{\nu}$ [GeV]')
ax_main.set_ylabel(r'Reco $E_{\nu}$ [GeV]')
ax_main.set_title(r'Neutrino energy: reco vs true (selected signal, no trigger, Pandora)')
ax_main.legend(fontsize=8.5, loc='upper left')
ax_main.set_xlim(0,3); ax_main.set_ylim(0,3)

# MPV residual panel 
ax_res.axhline(0, ls='--', lw=1.5, color='#555555', zorder=5)
ax_res.fill_between(E_TRUE_CEN[ok_b],
                    (r_mu-r_sg)[ok_b], (r_mu+r_sg)[ok_b],
                    color=BLUE, alpha=0.15, step='mid', zorder=2,
                    label=r'$\pm 1\sigma$ spread (RMS around ridge)')
ax_res.errorbar(E_TRUE_CEN[ok_p], r_mu[ok_p], yerr=r_err[ok_p],
                fmt='o', ms=6, lw=1.8, capsize=4, color=BLUE,
                markeredgecolor='white', markeredgewidth=0.5,
                label='MPV residual', zorder=5)
ax_res.set_xlabel(r'True $E_{\nu}$ [GeV]')
ax_res.set_ylabel(r'$(E_{MPV}-E_{true})/E_{true}$')
ax_res.set_title('MPV fractional residual vs energy')
ax_res.legend(fontsize=9)

# Resolution panel 
Ef_res = np.linspace(FIT_LO, FIT_HI, 200)
if fit_ok:
    ax_sg.plot(Ef_res, a_res+b_res*Ef_res, color=CRIMSON, lw=2.2, zorder=4,
               label=fr'Fit: $\sigma={a_res:.3f}\pm{perr_res[0]:.3f}'
                     fr'+({b_res:.3f}\pm{perr_res[1]:.3f})\,E_{{true}}$')
    ax_sg.fill_between(Ef_res,
        np.maximum(0,(a_res-perr_res[0])+(b_res-perr_res[1])*Ef_res),
        (a_res+perr_res[0])+(b_res+perr_res[1])*Ef_res,
        color=CRIMSON, alpha=0.18, zorder=3)

ax_sg.fill_between(E_TRUE_CEN[ok_b],
                   np.maximum(0,(r_sg-r_err)[ok_b]), (r_sg+r_err)[ok_b],
                   color=BLUE, alpha=0.18, step='mid', zorder=2)
ax_sg.plot(E_TRUE_CEN[ok_b], r_sg[ok_b], 'o', color=BLUE, ms=6, zorder=5,
           markeredgecolor='white', markeredgewidth=0.5,
           label='Resolution (RMS around ridge)')
ax_sg.set_xlabel(r'True $E_{\nu}$ [GeV]')
ax_sg.set_ylabel(r'$\sigma[(E_{reco}-E_{ridge})/E_{true}]$')
ax_sg.set_title('Energy resolution (RMS around ridge) vs energy')
ax_sg.set_ylim(bottom=0)
ax_sg.legend(fontsize=8)

#savefig(fig, 'energy_resolution.png')
plt.show()

### Method 3: Bifurcated Gaussian
A bifurcated Gaussian has a shared peak mu but independent widths
sigma_left and sigma_right on either side. This is the natural model for a distribution with a physical peak and an asymmetric tail: sigma_right captures the narrow rising edge of the distribution (well-reconstructed events near the true energy), while sigma_left captures the broader falling tail (events with significant energy loss). The MPV is simply mu from the fit, independent of the tail shape. This is more stable than the KDE because it has a well-defined optimisation target (chi-squared minimisation) and propagates a proper uncertainty on mu from the covariance matrix. The residuals become much more symmetric around zero because the fit is no longer contaminated by the tail. The remaining scatter is genuine statistical noise from the finite sample size (~30 events per bin), not a systematic bias in the method.

In [ ]:
from scipy.optimize import curve_fit
from scipy.stats import gaussian_kde

# ── Signal mask (inline to avoid variable clash) ──────────────────────────────
sig_mask = (df['is_signal']==1) & (df['true_electron_ke'].fillna(0) > 0.2)

ok = (np.array(sel_full(df), dtype=bool) &
      np.array(sig_mask, dtype=bool) &
      np.array(df['reco_nu_energy'].notna(), dtype=bool) &
      np.array(df['true_nu_energy'].notna(), dtype=bool) &
      np.array(df['reco_nu_energy'].values > 0, dtype=bool))

Et  = df['true_nu_energy'].values[ok]
Er  = df['reco_nu_energy'].values[ok]

print(f'N selected signal: {ok.sum()}')

# ── Bifurcated Gaussian ───────────────────────────────────────────────────────
def bifurcated_gaussian(x, mu, sig_l, sig_r, norm):
    return np.where(x < mu,
                    norm * np.exp(-0.5*((x-mu)/sig_l)**2),
                    norm * np.exp(-0.5*((x-mu)/sig_r)**2))

def fit_bifurcated(er_win, win_lo, win_hi, n_bins=14):
    """
    Fit bifurcated Gaussian. Returns (mu, mu_err, popt).
    Sanity-checks mu stays within the physical window.
    Falls back to KDE if fit fails or result is unphysical.
    """
    counts, edges = np.histogram(er_win, bins=n_bins,
                                 range=(win_lo, win_hi))
    cens  = 0.5*(edges[:-1]+edges[1:])
    errs  = np.maximum(np.sqrt(counts), 1.)
    mu0   = cens[np.argmax(counts)]
    sg0   = np.std(er_win) * 0.5
    norm0 = float(counts.max())

    fit_ok = False
    try:
        popt, pcov = curve_fit(
            bifurcated_gaussian, cens, counts,
            p0    = [mu0, sg0, sg0*1.5, norm0],
            sigma = errs, absolute_sigma=True,
            bounds=([win_lo, 0.01, 0.01, 0.],
                    [win_hi,  1.5,  1.5, 1e6]),
            maxfev=8000)
        mu_err = np.sqrt(max(pcov[0,0], 0))
        # sanity: mu must be inside the window and mu_err reasonable
        if (win_lo < popt[0] < win_hi) and (0 < mu_err < 0.5):
            return popt[0], mu_err, popt
    except Exception:
        pass

    # KDE fallback
    try:
        x_sc  = np.linspace(win_lo, win_hi, 300)
        kde   = gaussian_kde(er_win[(er_win>=win_lo)&(er_win<=win_hi)],
                             bw_method=0.3)
        mpv   = x_sc[np.argmax(kde(x_sc))]
        rng   = np.random.default_rng(42)
        boots = []
        for _ in range(80):
            samp = er_win[rng.integers(0, len(er_win), len(er_win))]
            samp = samp[(samp>=win_lo)&(samp<=win_hi)]
            if len(samp) > 3:
                k = gaussian_kde(samp, bw_method=0.3)
                boots.append(x_sc[np.argmax(k(x_sc))])
        mu_err = np.std(boots) if boots else 0.1
        if win_lo < mpv < win_hi:
            return mpv, mu_err, None
    except Exception:
        pass

    return np.nan, np.nan, None

# ── E_true bins ───────────────────────────────────────────────────────────────
FIT_LO, FIT_HI = 0.3, 2.2
E_TRUE_BINS = np.linspace(FIT_LO, FIT_HI, 12)
E_TRUE_CEN  = 0.5*(E_TRUE_BINS[:-1]+E_TRUE_BINS[1:])
N_BINS      = len(E_TRUE_CEN)
N_RECO_BINS = 16

peak_reco, peak_err = [], []
slice_data   = []
slice_counts = []
slice_edges  = []
slice_popt   = []

for lo, hi in zip(E_TRUE_BINS[:-1], E_TRUE_BINS[1:]):
    er_bin = Er[(Et>=lo) & (Et<hi)]
    slice_data.append(er_bin)
    if len(er_bin) > 8:
        expected = 0.5*(lo+hi)
        win_lo   = max(0.05, expected - 0.65)
        win_hi   = min(3.50, expected + 0.65)
        er_win   = er_bin[(er_bin>=win_lo) & (er_bin<=win_hi)]

        counts, edges = np.histogram(er_bin, bins=N_RECO_BINS,
                                     range=(win_lo, win_hi))
        slice_counts.append(counts)
        slice_edges.append(edges)

        if len(er_win) > 8:
            mpv, mpv_err, popt = fit_bifurcated(er_win, win_lo, win_hi,
                                                 n_bins=N_RECO_BINS)
        else:
            mpv, mpv_err, popt = np.nan, np.nan, None

        slice_popt.append(popt)
        peak_reco.append(mpv)
        peak_err.append(mpv_err)
    else:
        slice_counts.append(None); slice_edges.append(None)
        slice_popt.append(None)
        peak_reco.append(np.nan); peak_err.append(np.nan)

peak_reco = np.array(peak_reco)
peak_err  = np.array(peak_err)
ok_p      = ~np.isnan(peak_reco)

print(f'MPV found in {ok_p.sum()}/{N_BINS} bins')
for i in range(N_BINS):
    lo, hi = E_TRUE_BINS[i], E_TRUE_BINS[i+1]
    status = f'MPV={peak_reco[i]:.3f}+-{peak_err[i]:.3f}' if ok_p[i] else 'FAILED'
    method = 'bifurc.' if slice_popt[i] is not None else 'KDE'
    print(f'  [{lo:.2f},{hi:.2f}]: {status}  ({method})')

# ── Weighted linear fit ───────────────────────────────────────────────────────
weights  = 1.0 / np.where(peak_err[ok_p]>0, peak_err[ok_p]**2, 1e6)
weights /= weights.sum()

coeffs, cov_p = np.polyfit(E_TRUE_CEN[ok_p], peak_reco[ok_p], 1,
                            w=weights, cov=True)
m, c   = coeffs
m_err  = np.sqrt(cov_p[0,0])
c_err  = np.sqrt(cov_p[1,1])

y_pred   = m*E_TRUE_CEN[ok_p] + c
ss_res   = np.sum(weights*(peak_reco[ok_p]-y_pred)**2)
ss_tot   = np.sum(weights*(peak_reco[ok_p] -
                            np.average(peak_reco[ok_p],weights=weights))**2)
r_val_sq = 1 - ss_res/ss_tot

Ef   = np.linspace(FIT_LO, FIT_HI, 300)
Ep   = m*Ef + c
band = np.sqrt((Ef*m_err)**2 + c_err**2)

print(f'\nRidge fit: E_reco = {m:.4f}+-{m_err:.4f} * E_true {c:+.4f}+-{c_err:.4f}')
print(f'Weighted R^2 = {r_val_sq:.4f}')

# ── MPV-based residuals ───────────────────────────────────────────────────────
r_mu  = np.where(ok_p, (peak_reco - E_TRUE_CEN) / E_TRUE_CEN, np.nan)
r_err = np.where(ok_p, peak_err / E_TRUE_CEN, np.nan)

r_sg = []
for i, (lo, hi) in enumerate(zip(E_TRUE_BINS[:-1], E_TRUE_BINS[1:])):
    mask_bin   = (Et>=lo) & (Et<hi)
    er_b, et_b = Er[mask_bin], Et[mask_bin]
    if ok_p[i] and len(er_b) > 5:
        pred   = m*E_TRUE_CEN[i] + c
        spread = (er_b - pred) / np.where(et_b>0, et_b, 1)
        r_sg.append(np.std(spread))
    else:
        r_sg.append(np.nan)
r_sg = np.array(r_sg)

# ── Resolution fit ────────────────────────────────────────────────────────────
ok_b = ok_p & ~np.isnan(r_sg)
try:
    (a_res,b_res), cov_res = curve_fit(lambda x,a,b: a+b*x,
                                        E_TRUE_CEN[ok_b], r_sg[ok_b])
    perr_res = np.sqrt(np.diag(cov_res))
    fit_ok   = True
    print(f'Resolution: sigma = {a_res:.3f}+-{perr_res[0]:.3f} + ({b_res:.3f}+-{perr_res[1]:.3f})*E_true')
except Exception:
    fit_ok = False

# =============================================================================
# PLOT 1: per-slice diagnostic
# =============================================================================
ncols = 4
nrows = int(np.ceil(N_BINS / ncols))
fig_s, axes_s = plt.subplots(nrows, ncols, figsize=(15, 3.5*nrows),
                              gridspec_kw={'hspace':0.60,'wspace':0.38})
axes_s = axes_s.flatten()

for i, (lo, hi) in enumerate(zip(E_TRUE_BINS[:-1], E_TRUE_BINS[1:])):
    ax    = axes_s[i]
    er_b  = slice_data[i]
    label = fr'$E_{{true}}\in[{lo:.2f},{hi:.2f}]$ GeV'

    if slice_counts[i] is not None and len(er_b) > 8:
        counts = slice_counts[i]
        edges  = slice_edges[i]
        cens   = 0.5*(edges[:-1]+edges[1:])
        width  = edges[1]-edges[0]
        errs   = np.sqrt(np.maximum(counts, 1))

        ax.bar(cens, counts, width=width*0.85,
               color=BLUE, alpha=0.55, zorder=2)
        ax.errorbar(cens, counts, yerr=errs, fmt='none',
                    color=BLUE, lw=1.2, capsize=2, zorder=3)

        if slice_popt[i] is not None:
            x_fit = np.linspace(edges[0], edges[-1], 300)
            y_fit = bifurcated_gaussian(x_fit, *slice_popt[i])
            ax.plot(x_fit, y_fit, color=CRIMSON, lw=2, zorder=5,
                    label='Bifurc. fit')

        if ok_p[i]:
            mpv = peak_reco[i]
            ax.axvline(mpv, color=CRIMSON, lw=2, ls='--', zorder=6,
                       label=f'MPV={mpv:.2f} GeV')
            ax.axvspan(mpv-peak_err[i], mpv+peak_err[i],
                       color=CRIMSON, alpha=0.12, zorder=4)
            ridge_pred = m*E_TRUE_CEN[i] + c
            ax.axvline(ridge_pred, color=GOLD, lw=1.5, ls=':', zorder=6,
                       label=f'Ridge={ridge_pred:.2f} GeV')
            ax.legend(fontsize=6.5, loc='upper right')

        # enforce x-axis to physical window
        expected = 0.5*(lo+hi)
        ax.set_xlim(max(0, expected-0.75), min(3.5, expected+0.75))
        ax.text(0.04, 0.96, f'N={len(er_b)}', transform=ax.transAxes,
                fontsize=7, va='top', color='#444444')
    else:
        ax.text(0.5, 0.5, f'{label}\nInsufficient stats\n(N={len(er_b)})',
                ha='center', va='center', transform=ax.transAxes,
                fontsize=8, color='#888888')
        expected = 0.5*(lo+hi)
        ax.set_xlim(max(0, expected-0.75), min(3.5, expected+0.75))

    ax.set_title(label, fontsize=8)
    ax.set_xlabel(r'Reco $E_{\nu}$ [GeV]', fontsize=8)
    ax.set_ylabel('Events', fontsize=8)
    ax.tick_params(labelsize=7)

for j in range(i+1, len(axes_s)):
    axes_s[j].set_visible(False)

fig_s.suptitle(
    r'Reco $E_{\nu}$ per true $E_{\nu}$ slice'
    '\n(bifurcated Gaussian fit | crimson = MPV | gold = ridge prediction)',
    fontsize=11)
#savefig(fig_s, 'energy_slices.png')
plt.show()

# =============================================================================
# PLOT 2: main resolution figure
# =============================================================================
fig = plt.figure(figsize=(14,10))
gs  = fig.add_gridspec(2, 2, hspace=0.42, wspace=0.35)
ax_main = fig.add_subplot(gs[0,:])
ax_res  = fig.add_subplot(gs[1,0])
ax_sg   = fig.add_subplot(gs[1,1])

# main panel
h,xb,yb,im = ax_main.hist2d(Et, Er, bins=32, range=[[0,3],[0,3]], cmap=MANAGUA)
plt.colorbar(im, ax=ax_main, label='Events')
ax_main.plot([0,3],[0,3], 'w--', lw=1.8, label='Perfect reconstruction', zorder=5)
ax_main.errorbar(E_TRUE_CEN[ok_p], peak_reco[ok_p], yerr=peak_err[ok_p],
                 fmt='o', ms=5.5, color='white', markeredgecolor=CRIMSON,
                 markeredgewidth=1.8, capsize=3, lw=1.4, zorder=7,
                 label='Bifurc. Gaussian MPV per slice')
ax_main.plot(Ef, Ep, color=CRIMSON, lw=2.5, zorder=6,
             label=fr'Weighted MPV fit: $E_{{reco}}={m:.3f}\pm{m_err:.3f}'
                   fr'\,E_{{true}}{c:+.3f}\pm{c_err:.3f}$  ($R^2={r_val_sq:.3f}$)')
ax_main.fill_between(Ef, Ep-band, Ep+band, color=CRIMSON, alpha=0.20,
                     zorder=4, label=r'$1\sigma$ fit uncertainty')
ax_main.axvline(FIT_LO, ls=':', lw=1, color=CRIMSON, alpha=0.4)
ax_main.axvline(FIT_HI, ls=':', lw=1, color=CRIMSON, alpha=0.4)
ax_main.set_xlabel(r'True $E_{\nu}$ [GeV]')
ax_main.set_ylabel(r'Reco $E_{\nu}$ [GeV]')
ax_main.set_title(r'Neutrino energy: reco vs true (selected signal, no trigger, Pandora)')
ax_main.legend(fontsize=8.5, loc='upper left')
ax_main.set_xlim(0,3); ax_main.set_ylim(0,3)

# residual panel
ax_res.axhline(0, ls='--', lw=1.5, color='#555555', zorder=5)
ax_res.fill_between(E_TRUE_CEN[ok_b], (r_mu-r_sg)[ok_b], (r_mu+r_sg)[ok_b],
                    color=BLUE, alpha=0.15, step='mid', zorder=2,
                    label=r'$\pm 1\sigma$ spread around ridge')
ax_res.errorbar(E_TRUE_CEN[ok_p], r_mu[ok_p], yerr=r_err[ok_p],
                fmt='o', ms=6, lw=1.8, capsize=4, color=BLUE,
                markeredgecolor='white', markeredgewidth=0.5,
                label='MPV residual', zorder=5)
ax_res.set_xlabel(r'True $E_{\nu}$ [GeV]')
ax_res.set_ylabel(r'$(E_{MPV}-E_{true})/E_{true}$')
ax_res.set_title('MPV fractional residual vs energy')
ax_res.set_ylim(-0.5, 0.3)
ax_res.legend(fontsize=9)

# resolution panel
Ef_res = np.linspace(FIT_LO, FIT_HI, 200)
if fit_ok:
    ax_sg.plot(Ef_res, a_res+b_res*Ef_res, color=CRIMSON, lw=2.2, zorder=4,
               label=fr'Fit: $\sigma={a_res:.3f}\pm{perr_res[0]:.3f}'
                     fr'+({b_res:.3f}\pm{perr_res[1]:.3f})\,E_{{true}}$')
    ax_sg.fill_between(Ef_res,
        np.maximum(0,(a_res-perr_res[0])+(b_res-perr_res[1])*Ef_res),
        (a_res+perr_res[0])+(b_res+perr_res[1])*Ef_res,
        color=CRIMSON, alpha=0.18, zorder=3)

ax_sg.fill_between(E_TRUE_CEN[ok_b],
                   np.maximum(0,(r_sg-r_err)[ok_b]), (r_sg+r_err)[ok_b],
                   color=BLUE, alpha=0.18, step='mid', zorder=2)
ax_sg.plot(E_TRUE_CEN[ok_b], r_sg[ok_b], 'o', color=BLUE, ms=6, zorder=5,
           markeredgecolor='white', markeredgewidth=0.5,
           label='Resolution (RMS around ridge)')
ax_sg.set_xlabel(r'True $E_{\nu}$ [GeV]')
ax_sg.set_ylabel(r'$\sigma[(E_{reco}-E_{ridge})/E_{true}]$')
ax_sg.set_title('Energy resolution (RMS around ridge) vs energy')
ax_sg.set_ylim(0, 0.45)
ax_sg.legend(fontsize=8)

#savefig(fig, 'energy_resolution.png')
plt.show()

### Why residuals should be symmetric around zero
If the bifurcated Gaussian is correctly finding the physical peak of the reco energy distribution in each E_true bin, then by construction the residual (MPV - E_true_bin_centre)/E_true_bin_centre measures the
systematic energy scale offset. This should be a smooth, slowly varying
function of E_true (reflecting physical effects like missing neutron
energy) and the points should scatter symmetrically around this curve.
Any remaining asymmetry indicates either a fit convergence issue in a
specific bin (visible in the slice diagnostic plots) or genuine physical non-linearity in the energy response.

### Current status
The bifurcated Gaussian gives the most reliable MPV estimates and the
most symmetric residuals of the methods tried. The fit occasionally fails in low-statistics bins (N < 15) where the optimiser cannot constrain all four parameters simultaneously; these bins fall back to a KDE estimate. The energy scale (fit slope ~0.91) and resolution (~14-17%) are consistent with expectations for calorimetric reconstruction in LArTPC with missing hadronic energy.